In [1]:
import torchaudio
from torchaudio.pipelines import MMS_FA as _MMS_FA_BUNDLE
from pathlib import Path
import zipfile
import librosa
import torch
import numpy as np
from SMT import *
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm import tqdm
import json

In [2]:
subjects = [1272]
AUDIO_DIRS = [Path(f"./data/LibriSpeech/dev-clean/{str(subj)}") for subj in subjects]

norm_beta, norm_patches, utterance_bounds = calc_smt_on_corpus(AUDIO_DIRS[0], patch_length=100)

calc mels
patch utterances
apply kmeans
spectral decomp


In [7]:
# Broad phoneme classes matching MMS_FA's lowercase vocab
_CHAR_CLASS = {
    "a": "vowel",     "e": "vowel",     "i": "vowel",
    "o": "vowel",     "u": "vowel",
    "b": "stop",      "d": "stop",      "g": "stop",
    "k": "stop",      "p": "stop",      "t": "stop",
    "q": "stop",
    "f": "fricative", "s": "fricative", "v": "fricative",
    "x": "fricative", "z": "fricative", "c": "fricative",
    "m": "nasal",     "n": "nasal",
    "l": "liquid",    "r": "liquid",
    "h": "glide",     "j": "glide",     "w": "glide",     "y": "glide",
}
_CLASS_NAMES  = ["vowel", "stop", "fricative", "nasal", "liquid", "glide"]
_CLASS_COLORS = {
    "vowel":     "#4c72b0",
    "stop":      "#dd8452",
    "fricative": "#55a868",
    "nasal":     "#c44e52",
    "liquid":    "#937860",
    "glide":     "#da8bc3",
}

# ── 1.  Load MMS_FA forced-alignment model ────────────────────────────────────
print("Loading MMS_FA forced-alignment model ...")
_fa_model  = _MMS_FA_BUNDLE.get_model(with_star=False)
_fa_labels = _MMS_FA_BUNDLE.get_labels(star=None)
_fa_sr     = _MMS_FA_BUNDLE.sample_rate
_fa_dict   = {ch: i for i, ch in enumerate(_fa_labels)}
_fa_blank  = _fa_dict.get("-", 0)   # MMS_FA blank token index
print(f"  sr={_fa_sr} Hz  |  vocab={len(_fa_labels)}  |  blank='{_fa_labels[_fa_blank]}'(idx {_fa_blank})")
print(f"  Sample labels: {list(_fa_labels[:10])}")


Loading MMS_FA forced-alignment model ...
  sr=16000 Hz  |  vocab=28  |  blank='-'(idx 0)
  Sample labels: ['-', 'a', 'i', 'e', 'n', 'o', 'u', 't', 's', 'r']


In [8]:
def _align_and_extract(audio_np: np.ndarray, sample_rate: int,
                       transcript: str, n_mfcc: int = 40,
                       hop_ms: float = 10, win_ms: float = 20,
                       bin_ms: float = 100):
    """
    Forced-align one utterance and return MFCC vectors at fixed 100ms bins.
    Returns list of (char, broad_class, mean_mfcc_vec) in temporal order,
    where char/class are inferred from the dominant aligned token inside each bin.
    """
    if not isinstance(audio_np, np.ndarray) or audio_np.ndim != 1:
        return []
    if sample_rate != _fa_sr:
        audio_np = librosa.resample(audio_np, orig_sr=sample_rate, target_sr=_fa_sr)
    audio_np = audio_np.astype(np.float32)

    hop = int(round(_fa_sr * hop_ms / 1000))
    win = int(round(_fa_sr * win_ms / 1000))
    frames_per_bin = max(1, int(round(bin_ms / hop_ms)))

    mfcc_full = librosa.feature.mfcc(y=audio_np, sr=_fa_sr, n_mfcc=n_mfcc,
                                      hop_length=hop, win_length=win)
    n_frames = mfcc_full.shape[1]
    if n_frames == 0:
        return []

    token_ids = [_fa_dict[c] for c in transcript.lower() if c in _fa_dict]
    if len(token_ids) < 2:
        return []

    waveform = torch.from_numpy(audio_np).float().unsqueeze(0)
    with torch.no_grad():
        emissions, _ = _fa_model(waveform)
    emissions = torch.log_softmax(emissions, dim=-1)
    targets   = torch.tensor([token_ids], dtype=torch.int32)

    try:
        ali_t, ali_s = torchaudio.functional.forced_align(
            emissions, targets, blank=_fa_blank)
        segments = torchaudio.functional.merge_tokens(
            ali_t[0], ali_s[0], blank=_fa_blank)
    except Exception:
        return []

    ratio = n_frames / max(emissions.shape[1], 1)
    frame_chars = np.empty(n_frames, dtype=object)
    frame_chars[:] = None
    frame_cls = np.empty(n_frames, dtype=object)
    frame_cls[:] = None

    for seg in segments:
        ch = _fa_labels[seg.token]
        cls = _CHAR_CLASS.get(ch)
        if cls is None:
            continue
        s = min(int(seg.start * ratio), n_frames - 1)
        e = min(max(s + 1, int(seg.end * ratio)), n_frames)
        frame_chars[s:e] = ch
        frame_cls[s:e] = cls

    results = []
    for b_start in range(0, n_frames, frames_per_bin):
        b_end = min(n_frames, b_start + frames_per_bin)
        vec = mfcc_full[:, b_start:b_end].mean(axis=1)

        bin_chars = frame_chars[b_start:b_end]
        bin_cls = frame_cls[b_start:b_end]
        valid = [i for i, c in enumerate(bin_cls) if c is not None]
        if len(valid) == 0:
            continue

        valid_chars = [bin_chars[i] for i in valid]
        valid_cls = [bin_cls[i] for i in valid]
        labels, counts = np.unique(valid_chars, return_counts=True)
        ch = labels[np.argmax(counts)]
        cls = _CHAR_CLASS[ch] if ch in _CHAR_CLASS else valid_cls[0]
        results.append((ch, cls, vec))

    return results

In [9]:
audio_files = list_audio_files(AUDIO_DIRS[0])
librispeech_ds = torchaudio.datasets.LIBRISPEECH(
    root=str("/Users/zacariabalkhy/UCDAVIS_COURSES/EEC289A/final_project/data/"), url="dev-clean", download=True,
)

phon_vecs_list = []   # MFCC vector per 100ms bin
phon_cls_list  = []   # broad phoneme class per 100ms bin
phon_char_list = []   # dominant character per 100ms bin
phon_utt_list  = []   # utterance index (for trajectory analysis)
phon_pos_list  = []   # position within utterance (for trajectory analysis)
n_loaded = 0;  n_errors = 0


for idx in tqdm(range(len(audio_files)),
                desc="Aligning", leave=True):
    try:
        audio_raw, sr = librosa.load(str(audio_files[idx]), sr=None)
        transcript   = str(librispeech_ds.get_metadata(idx)[2])
        records      = _align_and_extract(audio_raw, sr, transcript)
        if not records:
            continue
        for pos, (ch, cls, vec) in enumerate(records):
            phon_vecs_list.append(vec)
            phon_cls_list.append(cls)
            phon_char_list.append(ch)
            phon_utt_list.append(n_loaded)
            phon_pos_list.append(pos)
        n_loaded += 1
    except Exception as exc:
        n_errors += 1
        if n_errors <= 3:
            print(f"  [warn] idx={idx}: {type(exc).__name__}: {exc}")

Aligning: 100%|██████████| 73/73 [00:20<00:00,  3.59it/s]


In [10]:
X_mfcc = np.stack(phon_vecs_list, axis=0).astype(np.float32)  # (N, n_mfcc)
y_cls  = np.array(phon_cls_list)
utt_arr = np.array(phon_utt_list)
pos_arr = np.array(phon_pos_list)

cls_counts = {c: int((y_cls == c).sum()) for c in _CLASS_NAMES if (y_cls == c).any()}
print(f"\n  {len(X_mfcc)} tokens from {n_loaded} utterances ({n_errors} skipped)")
print(f"  Class distribution: {cls_counts}")


  3377 tokens from 73 utterances (0 skipped)
  Class distribution: {'vowel': 1404, 'stop': 633, 'fricative': 486, 'nasal': 253, 'liquid': 271, 'glide': 330}


In [11]:
print(X_mfcc.shape)
print(norm_beta.shape)

(3377, 40)
(47505, 128)


In [ ]:
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from sklearn.metrics import silhouette_score, davies_bouldin_score

valid_cls = [c for c in _CLASS_NAMES if (y_cls == c).sum() >= 2]
mask_v    = np.isin(y_cls, valid_cls)
y_int     = np.array([_CLASS_NAMES.index(c) for c in y_cls[mask_v]])

def _cluster_metrics(X):
    Xv  = X[mask_v]
    sil = silhouette_score(Xv, y_int, metric="euclidean")
    db  = davies_bouldin_score(Xv, y_int)
    cls_ids   = np.unique(y_int)
    centroids = np.array([Xv[y_int == c].mean(axis=0) for c in cls_ids])
    intra  = np.mean([np.linalg.norm(Xv[y_int == c] - centroids[i], axis=1).mean()
                      for i, c in enumerate(cls_ids)])
    iis    = [np.linalg.norm(centroids[i]-centroids[j])
              for i in range(len(cls_ids)) for j in range(i+1, len(cls_ids))]
    return {"silhouette": sil,
            "davies_bouldin": db,
            "inter_intra": np.mean(iis) / (intra + 1e-8)}